# 3C · Groupby, Merge & Pivot — The Client Book
### Financial Analytics — Module 3

Three operations power almost every business analysis:

- **groupby** — "per segment / per city / per RM…" (split → apply → combine)
- **merge** — join two tables on a shared key (pandas' VLOOKUP, but correct)
- **pivot** — reshape long data into a report-ready grid

Dataset: 1,000 wealth-management clients + their transactions.

In [ ]:
import pandas as pd
import numpy as np

import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
clients = pd.read_csv(BASE + "client_book.csv", parse_dates=["onboard_date"])
print(clients.shape)
clients.head(3)

---
## groupby: split → apply → combine

*Split* the rows into groups, *apply* a calculation to each, *combine* the results into a table.

In [ ]:
# One measure, one grouping
clients.groupby("segment")["aum_inr"].mean().round(0)

In [ ]:
# Several measures at once with .agg()
summary = clients.groupby("segment").agg(
    clients=("client_id", "count"),
    total_aum=("aum_inr", "sum"),
    median_aum=("aum_inr", "median"),
    avg_products=("products_held", "mean"),
    churn_rate=("churned", "mean"),          # mean of 0/1 = the rate!
).round(2)

summary.sort_values("total_aum", ascending=False)

**Read the `churn_rate` column** — Mass churns at ~25%, Affluent at ~13%. You just found a real business pattern in two lines. That trick (`mean` of a 0/1 flag = a rate) is used constantly.

### The survivorship trap — live

The registry warns: churned clients remain in this file. Watch what happens if an analyst 'sensibly' filters to active clients first:

In [ ]:
avg_all = clients["aum_inr"].mean()
avg_active_only = clients[clients["churned"] == 0]["aum_inr"].mean()

print(f"Average AUM, all clients   : Rs {avg_all:,.0f}")
print(f"Average AUM, active only   : Rs {avg_active_only:,.0f}")
print(f"Difference                 : {(avg_active_only/avg_all - 1)*100:+.1f}%")

Filter first, and 'average client AUM' quietly changes. Neither number is *wrong* — but presenting the active-only figure as 'our average client' is **survivorship bias**, Module 1's second horseman, committed in one innocent-looking line.

### ✏️ Exercise 1
Group by `relationship_manager`: client count, total AUM, churn rate. Which RM manages the most AUM? Which has the worst churn?

In [ ]:
# your code here


---
## merge: joining two tables

Clients hold the *who*; transactions hold the *what*. Business questions need both. First, a fast version of 3B's cleaning (in real work you'd load the saved clean file):

In [ ]:
txn = pd.read_csv(BASE + "messy_transactions.csv").drop_duplicates()
txn = txn[txn["amount_inr"] > 0]
usd = txn["amount_inr"] < 20
txn.loc[usd, "amount_inr"] = txn.loc[usd, "amount_inr"] * 83.0
print(len(txn), "usable transactions")

In [ ]:
# Spend per customer, then JOIN onto the client book
spend = txn.groupby("customer_id")["amount_inr"].agg(
    total_spend="sum", txn_count="count").reset_index()

merged = clients.merge(
    spend,
    left_on="client_id",        # key in the LEFT table (clients)
    right_on="customer_id",     # key in the RIGHT table (spend)
    how="left",                 # keep ALL clients, even with no transactions
)
print(merged.shape)
merged[["client_id", "segment", "aum_inr", "total_spend", "txn_count"]].head()

**`how=` is the decision that changes your answer:**

| how | Keeps | Use when |
|---|---|---|
| `left` | every client, matched or not | "all clients, with spend where we have it" ✅ here |
| `inner` | only clients WITH transactions | analysis of transactors only — quietly drops everyone else |
| `outer` | everything from both | reconciliation: what's in one table but not the other? |

An `inner` join here would silently delete every non-transacting client — a self-inflicted survivorship bias. **Join type is an analytical decision, not a technicality.**

In [ ]:
# Who fell through? Clients with no transactions at all:
no_txn = merged["txn_count"].isna().sum()
print(f"{no_txn} clients have zero transactions -> their total_spend is NaN")

# For spend arithmetic, NaN means zero HERE (they truly spent nothing):
merged[["total_spend", "txn_count"]] = merged[["total_spend", "txn_count"]].fillna(0)

### ✏️ Exercise 2
Average `total_spend` per **segment**. Do wealthier segments transact more through this account? (Careful: does the answer change if you exclude zero-spend clients — and which number is the honest one to report?)

In [ ]:
# your code here


---
## pivot_table: the report grid

Long data is good for computers; grids are good for humans. `pivot_table` converts one to the other.

In [ ]:
grid = merged.pivot_table(
    index="city",              # rows
    columns="segment",         # columns
    values="aum_inr",          # what to aggregate
    aggfunc="sum",
).round(0)

# Order columns by wealth and show in Rs crore for readability
grid = grid[["Mass", "Affluent", "HNI", "Ultra-HNI"]] / 1e7
grid.round(1)

One line from Excel-pivot-table land, fully scripted and reproducible. Every number traceable to code — Module 1's lineage pillar again.

### ✏️ Exercise 3
Pivot: churn **rate** by `city` (rows) × `risk_profile` (columns). Hint: `values="churned", aggfunc="mean"`. Any city-profile pocket that looks alarming — and how many clients is it based on? (Add `aggfunc="count"` in a second pivot before you panic over a rate built on 3 people.)

In [ ]:
# your code here


---
## Recap

| Operation | Line | Answers |
|---|---|---|
| groupby + agg | `df.groupby("seg").agg(x=("col","mean"))` | "per segment…" |
| rate from a flag | `("churned", "mean")` | any % - of - group |
| merge | `a.merge(b, on=..., how="left")` | questions needing two tables |
| pivot_table | `df.pivot_table(index, columns, values, aggfunc)` | report grids |

**Next:** 3D — full EDA on NIFTY: returns, drawdowns, rolling volatility, and real data from yfinance.

---
*AI disclosure: ______*